# Árboles de Decisión para Clasificación

_Gini, entropía y comparación con la regresión logística_

**Módulo 1 — Aprendizaje Supervisado** | DSRP Machine Learning Engineering

![Aprendizaje Supervisado](assets/header.png)

## 1. Intuición

Igual que en regresión, dividimos el espacio en regiones rectangulares. La diferencia es que en cada hoja, en lugar de una media, predecimos la **clase mayoritaria** (o un vector de probabilidades estimadas como las frecuencias relativas dentro de la hoja). Más adelante visualizamos un árbol real entrenado sobre Telco Churn.

## 2. ¿Qué hace un "buen" split? — pureza

Queremos splits que dejen los hijos lo más **"puros"** posible: idealmente, todas las observaciones de un hijo de la misma clase. Para medir cuán impuro es un nodo se usan dos índices:

**Índice de Gini** (más usado, default en scikit-learn):

$$
G = 1 - \sum_{k=1}^{K} p_k^{2}
$$

**Entropía**:

$$
H = -\sum_{k=1}^{K} p_k \log_2 p_k
$$

donde $p_k$ es la proporción de la clase $k$ en el nodo. En un problema binario ambas alcanzan su **máximo cuando las clases están 50/50** (máxima incertidumbre) y valen **0 cuando el nodo es puro**:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p = np.linspace(0.001, 0.999, 200)
gini = 1 - p**2 - (1 - p)**2
ent  = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p, gini, label='Gini', lw=2)
ax.plot(p, ent,  label='Entropía', lw=2)
ax.set_xlabel('p (proporción de la clase 1)'); ax.set_ylabel('impureza')
ax.set_title('Gini y entropía en un nodo binario')
ax.legend(); ax.grid(True); plt.show()

El árbol busca el split que **más reduce la impureza** ponderando por el tamaño de los hijos. Eso se llama **ganancia de información** y es lo que decide qué variable y qué corte usar en cada nodo.

## 3. Logística vs árbol

| | Logística | Árbol |
|---|---|---|
| Frontera de decisión | lineal | escalonada (cajas) |
| No linealidades / interacciones | manuales | automáticas |
| Salida | probabilidades calibradas | proporciones por hoja |
| Interpretabilidad | coeficientes | reglas if/else |
| Sensibilidad a outliers | media | baja |
| Riesgo de overfitting | bajo (con regularización) | alto sin poda |

Visualmente, en un mismo problema 2D la logística traza una **recta** y el árbol traza **rectángulos**:

In [ ]:
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

Xs, ys = make_moons(n_samples=400, noise=0.25, random_state=0)
modelos = [
    ('Regresión logística', LogisticRegression().fit(Xs, ys)),
    ('Árbol (max_depth=4)', DecisionTreeClassifier(max_depth=4, random_state=0).fit(Xs, ys)),
]

xx, yy = np.meshgrid(
    np.linspace(Xs[:,0].min()-0.5, Xs[:,0].max()+0.5, 300),
    np.linspace(Xs[:,1].min()-0.5, Xs[:,1].max()+0.5, 300),
)
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (nombre, m) in zip(axes, modelos):
    zz = m.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5], colors=['#FFE4B5', '#B0E0E6'])
    ax.scatter(Xs[:,0], Xs[:,1], c=ys, cmap='coolwarm', edgecolor='k', s=25)
    ax.set_title(nombre)
plt.tight_layout(); plt.show()

## 4. Caso práctico — Telco Customer Churn

Mismo dataset que el notebook 04 (ver allí la descripción completa de columnas). Recordamos: 7.043 clientes con datos demográficos, contractuales y de facturación; el objetivo es predecir `Churn` (Yes/No). El loader convierte `TotalCharges` a numérico, elimina las filas con NaN y mapea `Churn` a 1/0.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
if not DATA.exists():
    raise FileNotFoundError(
        f'No se encontró {DATA}. Descarga el dataset desde '
        'https://www.kaggle.com/datasets/blastchar/telco-customer-churn '
        'y colócalo en data/ (ver README.md).'
    )

df = pd.read_csv(DATA)
# TotalCharges viene como string con espacios en clientes nuevos: lo limpiamos
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges']).reset_index(drop=True)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
print('Shape:', df.shape, '| Tasa de churn:', round(df.Churn.mean(), 3))
df.head()

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, classification_report,
)

sns.set_theme(style='whitegrid')

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
            'InternetService', 'Contract', 'PaperlessBilling', 'PaymentMethod']

X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, drop_first=True)
y = df['Churn']

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_tr.shape, X_te.shape

In [ ]:
tree = DecisionTreeClassifier(
    criterion='gini', max_depth=4, random_state=42,
).fit(X_tr, y_tr)

y_pred  = tree.predict(X_te)
y_proba = tree.predict_proba(X_te)[:, 1]

print(f'Accuracy  : {accuracy_score(y_te, y_pred):.3f}')
print(f'Precision : {precision_score(y_te, y_pred):.3f}')
print(f'Recall    : {recall_score(y_te, y_pred):.3f}')
print(f'F1        : {f1_score(y_te, y_pred):.3f}')
print(f'ROC AUC   : {roc_auc_score(y_te, y_proba):.3f}')

print('\n', classification_report(y_te, y_pred, target_names=['No churn', 'Churn']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_te, y_pred),
    display_labels=['No churn', 'Churn'],
).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Matriz de confusión — árbol')

RocCurveDisplay.from_predictions(y_te, y_proba, ax=axes[1])
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title('Curva ROC — árbol')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    tree,
    feature_names=X.columns,
    class_names=['No churn', 'Churn'],
    filled=True, rounded=True, ax=ax,
)
ax.set_title('Árbol de clasificación sobre Telco Churn (max_depth=4)')
plt.show()

In [ ]:
imp = (
    pd.Series(tree.feature_importances_, index=X.columns)
    .sort_values()
    .tail(12)
)
fig, ax = plt.subplots(figsize=(8, 5))
imp.plot.barh(ax=ax, color='teal')
ax.set_title('Importancia de variables — DecisionTreeClassifier')
plt.show()

In [ ]:
# --- Comparación contra logística sobre el mismo split ---
scaler = StandardScaler().fit(X_tr)
logit = LogisticRegression(max_iter=1000).fit(scaler.transform(X_tr), y_tr)
log_pred  = logit.predict(scaler.transform(X_te))
log_proba = logit.predict_proba(scaler.transform(X_te))[:, 1]

comparativa = pd.DataFrame({
    'modelo':   ['Decision Tree', 'Regresión logística'],
    'accuracy': [accuracy_score(y_te, y_pred), accuracy_score(y_te, log_pred)],
    'f1':       [f1_score(y_te, y_pred),       f1_score(y_te, log_pred)],
    'roc_auc':  [roc_auc_score(y_te, y_proba), roc_auc_score(y_te, log_proba)],
})
comparativa

In [ ]:
# --- Curva de complejidad (max_depth) ---
depths = range(1, 16)
tr, te = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_tr, y_tr)
    tr.append(accuracy_score(y_tr, m.predict(X_tr)))
    te.append(accuracy_score(y_te, m.predict(X_te)))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(depths, tr, marker='o', label='Train accuracy')
ax.plot(depths, te, marker='o', label='Test  accuracy')
ax.set_xlabel('max_depth'); ax.set_ylabel('accuracy')
ax.set_title('Curva de complejidad — DecisionTreeClassifier')
ax.legend(); plt.show()

## 5. Referencias

- Breiman et al. (1984). *Classification and Regression Trees*.
- Quinlan, J. R. (1986). *Induction of Decision Trees*.
- ISLR cap. 8.
- scikit-learn user guide: https://scikit-learn.org/stable/modules/tree.html
- Dataset: https://www.kaggle.com/datasets/blastchar/telco-customer-churn